# Using Maths in EasyScience

EasyScience objects are numbers that know their **unit**, their **uncertainty** and, for a
`Parameter`, the **bounds** a fit may explore. This tutorial is a series of short examples of how maths
with them works, and of how it is expected to behave:

* the fundamental operations `+`, `-`, `*`, `/`, `**`, negation and `abs`;
* the functions of `easyscience.math` (`es.math`): exponential, logarithm, trigonometric, hyperbolic
  and error functions.

The `es.math` examples are themed on a small diffraction experiment (Bragg's law, a Debye-Waller factor,
a slit), but each one stands alone. Every result is a **new** object. Its value and unit come from the
calculation, its variance is propagated from the variances of the inputs, and its bounds are the exact
range the result can reach while the inputs stay inside their bounds.

No data and no fit are needed until the last section, where the same functions are applied to the result
of a fit.

In [1]:
import numpy as np
import plotly.graph_objects as go

import easyscience as es
from easyscience import DescriptorNumber
from easyscience import Parameter

## Numbers with an uncertainty

A `Parameter` has a value, a unit, an uncertainty, and bounds. Here is a length of 2 m with an
uncertainty of 0.2 m, which a fit may vary between 0 and 10 m:

In [2]:
length = Parameter(name='length', value=2.0, unit='m', variance=0.04, min=0.0, max=10.0)
length

<Parameter 'length': 2.0000 ± 0.2000 m, bounds=[0.0:10.0]>

The uncertainty is stored as a **variance**, and `error` is its square root:

In [3]:
print(length.value, length.unit)
print(length.variance, length.error)
print(length.min, length.max)

2.0 m
0.04 0.2
0.0 10.0


A `DescriptorNumber` is the same without the fitting bounds. It is the right choice for a number that
is known but is never fitted, such as a measured constant:

In [4]:
scale = DescriptorNumber(name='scale', value=3.0, variance=0.01)
scale

<DescriptorNumber 'scale': 3.0000 ± 0.1000>

The results of `es.math` have many digits in their bounds, so we define a helper that prints one
tidy line per number: value, error, unit and, for a `Parameter`, the bounds.

In [5]:
def show(number):
    """One line with value, error, unit and, for a Parameter, the bounds."""
    text = f'{number.value:.4f} ± {number.error:.4f} {number.unit}'
    if hasattr(number, 'min'):
        text += f'   in [{number.min:.4f}, {number.max:.4f}]'
    return text


print(show(length))
print(show(scale))

2.0000 ± 0.2000 m   in [0.0000, 10.0000]
3.0000 ± 0.1000 dimensionless


## Basic arithmetic

The operators work between `Parameter`s, `DescriptorNumber`s and plain numbers. The result is a new
object: a `Parameter` (with bounds) if any operand is a `Parameter`, and a `DescriptorNumber`
otherwise.

For independent inputs, the variance of a result is propagated to first order,

$$
\sigma_f^2 = \sum_i \left(\frac{\partial f}{\partial x_i}\right)^2 \sigma_i^2 .
$$

### Addition and subtraction

Only numbers with the same physical dimension can be added. If the units differ, the second operand
is converted to the unit of the first, so the result is in metres here:

In [6]:
width = Parameter(name='width', value=50.0, unit='cm', variance=4.0, min=10.0, max=100.0)
total = length + width
print(show(total))

2.5000 ± 0.2010 m   in [0.1000, 11.0000]


The variances add, $0.2^2 + 0.02^2 = 0.0404\ \text{m}^2$, and so do the bounds:
$[0 + 0.1,\ 10 + 1.0]$ m. Subtraction combines the bounds the other way round: the lowest possible
difference is the minimum of the first minus the maximum of the second.

In [7]:
difference = length - width
print(show(difference))

1.5000 ± 0.2010 m   in [-1.0000, 9.9000]


A plain number can only be added to a dimensionless value. Adding one to a length is an error:

In [8]:
try:
    length + 1.0
except Exception as error:
    print(f'{type(error).__name__}: {error}')

UnitError: Numbers can only be added to dimensionless values


### Multiplication and division

Units multiply and divide. The area of a rectangle, with the sides in the same unit:

In [9]:
height = Parameter(name='height', value=1.5, unit='m', variance=0.01, min=1.0, max=3.0)
area = length * height
print(show(area))

3.0000 ± 0.3606 m^2   in [0.0000, 30.0000]


The bounds of a product are the lowest and highest of the four products of the bounds. A speed is a
length over a time:

In [10]:
time = Parameter(name='time', value=4.0, unit='s', variance=0.0625, min=2.0, max=8.0)
speed = length / time
print(show(speed))

0.5000 ± 0.0590 m/s   in [0.0000, 5.0000]


Multiplying a `DescriptorNumber` by a plain number gives a `DescriptorNumber`, so there are no
bounds:

In [11]:
doubled = scale * 2
print(type(doubled).__name__, show(doubled))

DescriptorNumber 6.0000 ± 0.2000 dimensionless


### Powers, negation and absolute value

The exponent of a power is a plain number. Negation flips the bounds, and `abs` folds them:

In [12]:
print(show(length**2))
print(show(-length))
print(show(abs(-length)))

4.0000 ± 0.8000 m^2   in [0.0000, 100.0000]
-2.0000 ± 0.2000 m   in [-10.0000, -0.0000]
2.0000 ± 0.2000 m   in [0.0000, 10.0000]


## Maths functions with es.math

`es.math` provides the functions that the operators do not cover. Each one takes a `Parameter` or a
`DescriptorNumber` and returns a new one with the value, unit, variance and bounds worked out:

| family | functions |
|---|---|
| exponential and logarithm | `exp`, `log`, `log10`, `sqrt` |
| trigonometric | `sin`, `cos`, `tan`, `sinc` |
| inverse trigonometric | `arcsin`, `arccos`, `arctan`, `arctan2` (aliases `asin`, `acos`, `atan`, `atan2`) |
| hyperbolic | `sinh`, `cosh`, `tanh`, `arcsinh`, `arccosh`, `arctanh` (aliases `asinh`, `acosh`, `atanh`) |
| error function | `erf`, `erfc` |

They are used as `es.math.sin(angle)`. The value and the variance come from Scipp, which also checks
the units (for `erf`, `erfc` and `arctan2`, which Scipp cannot propagate a variance through, `es.math`
supplies the derivative itself); the bounds are the exact range of the function over the bounds of the
input.

### Exponential, logarithm and square root

These functions take a dimensionless number, except `sqrt`, which also takes a number whose unit can be
halved. The Debye-Waller factor of a crystal, $\exp\!\left(-q^2\langle u^2\rangle/3\right)$, combines a
scattering vector $q$ and a mean-square displacement $\langle u^2\rangle$; the units cancel, so `exp`
accepts it:

In [13]:
q_par = Parameter(name='q', value=2.0, unit='1/angstrom', variance=0.02**2, min=0.5, max=4.0)
u2 = Parameter(name='u2', value=0.01, unit='angstrom**2', variance=0.002**2, min=0.0, max=0.05)

debye_waller = es.math.exp(-(q_par**2) * u2 / 3)
print(show(debye_waller))

0.9868 ± 0.0026 dimensionless   in [0.7659, 1.0000]


The bounds are the exact range of the function over the bounds of the inputs, here $[0.77,\ 1]$. A
relative intensity in decibels uses `log10`, and the length of a vector from two components uses `sqrt`:

In [14]:
intensity = Parameter(name='intensity', value=250.0, variance=100.0, min=100.0, max=400.0)
reference = DescriptorNumber(name='reference', value=1000.0, variance=0.0)
decibel = 10 * es.math.log10(intensity / reference)
print(show(decibel))

a = Parameter(name='a', value=3.0, variance=0.01, min=2.0, max=4.0)
b = Parameter(name='b', value=4.0, variance=0.01, min=3.0, max=5.0)
amplitude = es.math.sqrt(a**2 + b**2)
print(show(amplitude))

-6.0206 ± 0.1737 dimensionless   in [-10.0000, -3.9794]
5.0000 ± 0.1000 dimensionless   in [3.6056, 6.4031]


The square root of an area is a length:

In [15]:
print(show(es.math.sqrt(area)))

1.7321 ± 0.1041 m   in [0.0000, 5.4772]


**Expected behaviour: a round trip returns where it started.** The derivative of $\exp(\log x)$ is 1,
so the value *and* the uncertainty come back unchanged:

In [16]:
x = Parameter(name='x', value=1.2, variance=0.0025, min=0.5, max=3.0)
round_trip = es.math.exp(es.math.log(x))
print(show(round_trip))
print(np.isclose(round_trip.variance, x.variance))

1.2000 ± 0.0500 dimensionless   in [0.5000, 3.0000]
True


### Trigonometric functions

Bragg's law relates the spacing $d$ of the lattice planes to the scattering angle $\theta$ and the
wavelength $\lambda$, and the scattering vector is $q$:

$$
d = \frac{\lambda}{2\sin\theta}, \qquad q = \frac{4\pi\sin\theta}{\lambda}.
$$

The argument of `sin`, `cos` and `tan` must have an angle unit, `rad` or `deg`; the result is
dimensionless. The uncertainty is converted correctly: $\sin 15^\circ$ has an error of
$\cos 15^\circ \times \sigma_\theta$ with $\sigma_\theta$ in radians.

In [17]:
theta = Parameter(name='theta', value=15.0, unit='deg', variance=0.1**2, min=5.0, max=60.0)
wavelength = DescriptorNumber(name='wavelength', value=1.5406, unit='angstrom', variance=0.0005**2)

s = es.math.sin(theta)
print(show(s))

0.2588 ± 0.0017 dimensionless   in [0.0872, 0.8660]


The wavelength is a measured constant, so it is a `DescriptorNumber` (no bounds); the angle may be
fitted, so it is a `Parameter`. Ordinary arithmetic then gives $d$ and $q$, with their uncertainties and
bounds:

In [18]:
d = wavelength / (2 * s)
q = 4 * np.pi * s / wavelength
print(show(d))
print(show(q))

2.9762 ± 0.0194 Å   in [0.8895, 8.8382]
2.1111 ± 0.0138 1/Å   in [0.7109, 7.0640]


In [19]:
for function in (es.math.cos, es.math.tan):
    print(f'{function.__name__:4}', show(function(theta)))

cos  0.9659 ± 0.0005 dimensionless   in [0.5000, 0.9962]
tan  0.2679 ± 0.0019 dimensionless   in [0.0875, 1.7321]


`sinc(x)` is $\sin(x)/x$, not the normalised $\sin(\pi x)/(\pi x)$ of NumPy. It takes `rad` or a
dimensionless number. The intensity of single-slit diffraction is proportional to $\mathrm{sinc}^2$:

In [20]:
x = Parameter(name='x', value=1.2, variance=0.0025, min=0.5, max=3.0)
print(show(es.math.sinc(x) ** 2))

0.6033 ± 0.0268 dimensionless   in [0.0022, 0.9194]


### Inverse trigonometric functions

The inverse functions return an angle, in radians by default; the `unit` keyword chooses `'deg'`.
**Expected behaviour:** the Bragg angle recovered from $d$ and $\lambda$ is the angle we started with:

In [21]:
recovered = es.math.arcsin(wavelength / (2 * d), unit='deg')
print(show(recovered))
print(show(es.math.arccos(es.math.cos(theta))))

15.0000 ± 0.1002 deg   in [5.0000, 60.0000]
0.2618 ± 0.0017 rad   in [0.0873, 1.0472]


The value is the one we started with, and the error is almost the same: 0.1002 instead of 0.1000. The
small excess is explained in *Limits of first-order propagation* below.

`arctan2(y, x)` uses the signs of both arguments to find the quadrant. The phase of a structure factor
$F = a + \mathrm{i}\,b$ is `arctan2(b, a)`; the two arguments must have the same unit (here none):

In [22]:
phase = es.math.arctan2(b, a, unit='deg')
print(show(phase))

53.1301 ± 1.1459 deg   in [36.8699, 68.1986]


The shorter names `asin`, `acos`, `atan` and `atan2` are aliases of the same functions:

In [23]:
es.math.asin is es.math.arcsin

True

### Hyperbolic functions

`sinh`, `cosh` and `tanh` and their inverses take a dimensionless number. The reduced magnetisation of a
two-level paramagnet is $\tanh$ of the reduced field, and `arctanh` undoes it:

In [24]:
reduced_field = Parameter(name='reduced_field', value=0.5, variance=0.0004, min=0.0, max=1.0)
magnetisation = es.math.tanh(reduced_field)
print(show(magnetisation))
print(show(es.math.arctanh(magnetisation)))

0.4621 ± 0.0157 dimensionless   in [0.0000, 0.7616]
0.5000 ± 0.0200 dimensionless   in [0.0000, 1.0000]


The bounds of `cosh` show why the exact range matters: $\cosh$ has its minimum at 0, so a range that
includes 0 does not start at the value of $\cosh$ at the lower bound.

In [25]:
z = Parameter(name='z', value=0.5, variance=0.0025, min=-1.0, max=2.0)

for function in (es.math.sinh, es.math.cosh):
    print(f'{function.__name__:5}', show(function(z)))

sinh  0.5211 ± 0.0564 dimensionless   in [-1.1752, 3.6269]
cosh  1.1276 ± 0.0261 dimensionless   in [1.0000, 3.7622]


### The error function

The fraction of a Gaussian beam of width $\sigma$ that passes a slit of half-width $a$ is
$\mathrm{erf}\!\left(a/(\sigma\sqrt{2})\right)$, and `erfc` $= 1 - \mathrm{erf}$ is the fraction that is
blocked:

In [26]:
slit = Parameter(name='slit', value=2.0, unit='mm', variance=0.1**2, min=1.0, max=3.0)
beam = Parameter(name='beam', value=1.5, unit='mm', variance=0.05**2, min=1.0, max=2.0)

transmitted = es.math.erf(slit / (beam * np.sqrt(2)))
blocked = es.math.erfc(slit / (beam * np.sqrt(2)))
print(show(transmitted))
print(show(blocked))
print(transmitted.value + blocked.value)

0.8176 ± 0.0263 dimensionless   in [0.3829, 0.9973]
0.1824 ± 0.0263 dimensionless   in [0.0027, 0.6171]
1.0


### Parameters and DescriptorNumbers as input

The functions follow the same rule as the operators: the result is a `Parameter` if any argument is a
`Parameter`, and a `DescriptorNumber` otherwise. A `DescriptorNumber` has no bounds, so it counts as a
single point when the bounds of a result are worked out. In the Bragg example, `wavelength` is a
`DescriptorNumber` and `theta` a `Parameter`, so `d` is a `Parameter`:

In [27]:
print(type(wavelength).__name__, type(theta).__name__, '->', type(d).__name__)

constant = DescriptorNumber(name='constant', value=0.5, variance=0.0004)
result = es.math.exp(constant)
print(type(result).__name__, show(result))

DescriptorNumber Parameter -> Parameter
DescriptorNumber 1.6487 ± 0.0330 dimensionless


## Uncertainty and bounds of a result

Because a result keeps its uncertainty, we can draw a function together with the band that the
uncertainty of its argument gives. `es.math` works on one number at a time, so we loop over the angles.
Here is the lattice spacing $d(\theta)$ of the Bragg example, for an angle with an uncertainty of
$1^\circ$ (ten times larger than above, so that the band can be seen). The chart is interactive: hover
over the curve to read the values. It loads Plotly from the internet, so it needs a connection. The charts are shown with a small helper,
`show_figure(fig)`, which is defined in a hidden cell; in your own notebook `fig.show()` does the same.

In [28]:
import base64

from IPython.display import IFrame
from IPython.display import display


def show_figure(fig: go.Figure, height: int = 420) -> None:
    """Show a Plotly figure as an embedded iframe, so that it renders in the docs."""
    page = (
        go
        .Figure(fig)
        .update_layout(template='plotly_white')
        .to_html(
            full_html=True,
            include_plotlyjs='cdn',
            default_width='100%',
            default_height=f'{height - 20}px',
        )
    )
    encoded = base64.b64encode(page.encode('utf-8')).decode('ascii')
    display(IFrame(src=f'data:text/html;base64,{encoded}', width='100%', height=height))

In [29]:
angles = np.linspace(5.0, 60.0, 60)

values, errors = [], []
for value in angles:
    angle = Parameter(name='theta', value=value, unit='deg', variance=1.0**2)
    spacing = wavelength / (2 * es.math.sin(angle))
    values.append(spacing.value)
    errors.append(spacing.error)
values, errors = np.array(values), np.array(errors)

fig = go.Figure()
fig.add_trace(
    go.Scatter(x=angles, y=values - errors, mode='lines', line=dict(width=0), showlegend=False)
)
fig.add_trace(
    go.Scatter(
        x=angles,
        y=values + errors,
        mode='lines',
        line=dict(width=0),
        fill='tonexty',
        fillcolor='rgba(99, 110, 250, 0.25)',
        name='± 1 error',
    )
)
fig.add_trace(
    go.Scatter(
        x=angles,
        y=values,
        mode='lines',
        name='d',
        line=dict(color='rgb(99, 110, 250)'),
        customdata=errors,
        hovertemplate='theta = %{x:.1f} deg<br>d = %{y:.3f} ± %{customdata:.3f} Å<extra></extra>',
    )
)
fig.update_layout(
    xaxis_title='theta / deg',
    yaxis_title='d / Å',
    margin=dict(l=60, r=20, t=30, b=50),
    legend=dict(x=0.6, y=0.95),
)
show_figure(fig)

The band is widest at small angles: the relative error of $d$ is $\cot\theta\,\sigma_\theta$.

The bounds of the input carry through as well. The sine of an angle that may vary between $0^\circ$ and
$180^\circ$ reaches 1 in the middle, so the upper bound of the result is exactly 1, not the sine of
either end:

In [30]:
wide = Parameter(name='wide', value=80.0, unit='deg', variance=1.0, min=0.0, max=180.0)
print(show(es.math.sin(wide)))

0.9848 ± 0.0030 dimensionless   in [0.0000, 1.0000]


## Limits of first-order propagation

The variance is correct to first order. That is excellent when the uncertainty is small compared with
how fast the function changes, and it has limits that are good to know.

**Bad inputs are refused with an error.** A value outside the domain of a function raises an error
that names the parameter, a wrong unit raises a unit error, and a `tan` too close to a pole is refused:

In [31]:
cases = {
    'log10 of zero': lambda: es.math.log10(Parameter(name='v', value=0.0, variance=0.01)),
    'arcsin of 1.2 (no Bragg angle exists)': lambda: es.math.arcsin(
        Parameter(name='v', value=1.2, variance=0.01)
    ),
    'sqrt of a length': lambda: es.math.sqrt(
        Parameter(name='v', value=2.0, unit='m', variance=0.01)
    ),
    'tan at a pole': lambda: es.math.tan(
        Parameter(name='v', value=np.pi / 2, unit='rad', variance=0.01)
    ),
}
for label, build in cases.items():
    try:
        build()
    except Exception as error:
        print(f'{label}: {type(error).__name__}: {error}\n')

log10 of zero: ValueError: The value of 'v' (0.0) is outside the domain (0.0, inf] of log10().

arcsin of 1.2 (no Bragg angle exists): ValueError: The value of 'v' (1.2) is outside the domain [-1.0, 1.0] of arcsin().

sqrt of a length: UnitError: Unsupported unit as result of sqrt: sqrt(m).

tan at a pole: ValueError: The value of 'v' (1.5707963267948966 rad) is within 0.1 rad of a tan asymptote (pi/2 + n*pi); the propagated variance would explode.



`arcsin` of a ratio larger than 1 is the Bragg case $\lambda/(2d) > 1$: no angle satisfies the law, and
the function says so instead of returning a meaningless number.

**The same number used twice is treated as two independent numbers.** In the round trip above,
`wavelength` appears twice, in $d$ and again in the `arcsin`. Mathematically the wavelength cancels and
the variance must come back as exactly that of `theta`, but the two uses are treated as independent, so
a little extra variance is added:

In [32]:
print(f'variance after the round trip: {recovered.variance:.6f}')
print(f'variance of theta:             {theta.variance:.6f}')

variance after the round trip: 0.010050
variance of theta:             0.010000


The difference is small here because the wavelength is known very precisely, but it grows with the
uncertainty of the number that is used twice.

**Where the function is flat, first order gives zero.** At the top of the sine the derivative is 0, so
the first-order error is 0, although the argument is uncertain and the result really varies a little. A
Monte Carlo sample of the same sine shows the difference:

In [33]:
sigma = 0.3
angles = np.linspace(0.0, 2 * np.pi, 80)

first_order = np.array([
    es.math.sin(Parameter(name='angle', value=value, unit='rad', variance=sigma**2)).error
    for value in angles
])

rng = np.random.default_rng(seed=1)
samples = rng.normal(angles[:, None], sigma, size=(len(angles), 20_000))
monte_carlo = np.sin(samples).std(axis=1)

fig = go.Figure()
fig.add_trace(go.Scatter(x=angles, y=first_order, mode='lines', name='first order (es.math)'))
fig.add_trace(
    go.Scatter(x=angles, y=monte_carlo, mode='lines', line=dict(dash='dash'), name='Monte Carlo')
)
fig.update_layout(
    xaxis_title='angle / rad',
    yaxis_title='error of sin(angle)',
    margin=dict(l=60, r=20, t=60, b=50),
    legend=dict(orientation='h', x=0.5, xanchor='center', y=1.1),
)
show_figure(fig, height=380)

The two agree to within a few per cent away from the extremes. At $\pi/2$ and $3\pi/2$ the first-order
error drops to (almost) zero, while the Monte Carlo error stays near $\sigma^2/\sqrt{2}\approx 0.06$.
The same kind of failure happens when the uncertainty is large compared with the scale of the function
(for example `exp` of a number with an uncertainty of 1), and close to a pole of `tan`.

**Two different inputs are treated as independent.** `a - b` and `arctan2(y, x)` combine the variances of
the two inputs as if they were unrelated. If the inputs are correlated, for example the slope and the
intercept of the same fit, the variance of a function of both can be off by a large factor.

## Maths on the result of a fit

`es.math` works on single numbers, not on arrays. Inside the model function of a fit, which is called
on a whole array of $x$ values, use NumPy on the `.value` of the parameters. After the fit, use
`es.math` on the fitted parameters: they carry the uncertainty that the fit found, so a derived
quantity comes with its own uncertainty.

We fit a straight line to simulated data, and ask for the angle of the line:

In [34]:
from easyscience import Fitter
from easyscience.base_classes import ModelBase

In [35]:
class StraightLine(ModelBase):
    """y = slope * x + intercept"""

    def __init__(self, slope: float, intercept: float):
        super().__init__()
        self._slope = Parameter(name='slope', value=slope)
        self._intercept = Parameter(name='intercept', value=intercept)

    @property
    def slope(self) -> Parameter:
        return self._slope

    @property
    def intercept(self) -> Parameter:
        return self._intercept

    def __call__(self, x):
        return self._slope.value * x + self._intercept.value

In [36]:
rng = np.random.default_rng(seed=20261013)
noise = 0.3
x = np.linspace(0.0, 10.0, 40)
y = 0.6 * x + 1.0 + rng.normal(scale=noise, size=x.size)
weights = np.full_like(x, 1.0 / noise)

model = StraightLine(slope=0.1, intercept=0.0)
fitter = Fitter(model, model)
result = fitter.fit(x, y, weights=weights)

print(show(model.slope))
print(show(model.intercept))

0.6009 ± 0.0136 dimensionless   in [-inf, inf]
1.0138 ± 0.0791 dimensionless   in [-inf, inf]


The angle of the line with the $x$ axis is the arctangent of the slope. `arctan` needs a dimensionless argument, and the slope is one; the `unit` keyword then returns the
angle in degrees:

In [37]:
line_angle = es.math.arctan(model.slope, unit='deg')
print(show(line_angle))

31.0029 ± 0.5734 deg   in [-90.0000, 90.0000]


The uncertainty of the angle comes from the uncertainty of the fitted slope, by the same first-order
rule as above.

## Summary

* `Parameter` and `DescriptorNumber` carry a value, a unit and a variance; a `Parameter` also carries
  bounds. Arithmetic and `es.math` functions return **new** objects.
* The units are checked and converted, the variance is propagated to first order for independent
  inputs, and the bounds are the exact range of the result.
* `es.math` covers exponential, logarithmic, trigonometric, inverse trigonometric, hyperbolic and
  error functions, with a `Parameter` or a `DescriptorNumber` as input.
* A value outside the domain of a function raises an error. The first-order variance is unreliable for
  large uncertainties, at a flat spot of the function, near a pole, and for inputs that are correlated
  or that are the same number used twice.
* Use `es.math` on fitted parameters, after the fit, to get a derived quantity with its uncertainty.

See the [es.math reference](../api-reference/math.md) for every function.